# AI Hardware Architecture

**Author:** Nitin Sarangdhar  
**Affiliation:** SemiX Visiting Professor of practice | IIT Bombay  
**Date:** March 2026  

# Lab: From Pixels to Predictions (A Hardware-Centric CNN Pipeline)

### Objective:
In this lesson, we break down the **Inference Pipeline** of a Convolutional Neural Network (CNN). Unlike the fully-connected XOR network, a CNN uses **Spatially-Aware** operations to detect patterns like edges before making a final classification.

---

### The 5 Stages of the AI Hardware Pipeline:

1. **Convolution (Feature Extraction):** - A $3 \times 3$ **Kernel** (Filter) slides over the $10 \times 10$ input image.
   - **Hardware Insight:** This is a series of **Multiply-Accumulate (MAC)** operations. In an NPU/TPU, we aim for high "Data Reuse" to minimize power-hungry memory reads.

2. **ReLU Activation (Non-Linearity):**
   - We apply $f(x) = \max(0, x)$ to "clean" the feature map.
   - **Hardware Insight:** This is the "cheapest" operation in silicon (a simple sign-bit check). It introduces **Sparsity**, allowing hardware to potentially skip zero-value computations in later stages.

3. **Max Pooling (Spatial Compression):**
   - We reduce the $8 \times 8$ feature map to a $4 \times 4$ grid by taking the maximum value in $2 \times 2$ neighborhoods.
   - **Hardware Insight:** Pooling reduces the **Memory Footprint** and makes the system robust to small shifts in the input (Spatial Invariance).

4. **Flattening (The Memory Bridge):**
   - We transition from a 2D "Spatial Map" to a 1D "Feature Vector."
   - **Hardware Insight:** This is a **Memory Management** step. We stop using 2D (Row/Col) addressing and treat the SRAM as a contiguous linear buffer for high-speed streaming.

5. **Classification (The 'Brain' Step):**
   - We perform a **Dot Product** between our 1D Feature Vector and a learned **Weight Template**.
   - **Hardware Insight:** This is a **Vector-Matrix Multiplication (VMM)**. The "Winner" (highest score) represents the network's final decision.

---
> **Class Discussion:** Why do we use a $3 \times 3$ kernel instead of a $10 \times 10$ kernel?
> *Answer: Weight sharing! A $3 \times 3$ kernel uses only 9 parameters regardless of image size, drastically reducing the transistor count and power leakage on-chip.*

The CNN classifies an input image as a vertical bar, a horizontal bar or neither of the two.

In [1]:
import numpy as np

# --- 1. SETUP THE WORLD ---
image = np.zeros((10, 10))
image[:, 4:6] = 10  # A bright vertical bar (intensity 10)

kernel = np.array([
    [-1, 0, 1],
    [-1, 0, 1],
    [-1, 0, 1]
])

# --- 2. DATA OVERVIEW ---
print("="*60)
print("STEP 1: THE INPUT DATA & KERNEL")
print("="*60)
print("\nFULL 10x10 INPUT IMAGE:")
for row in image:
    print(" ".join(f"{int(val):3d}" for val in row))

print("\n3x3 CONVOLUTION KERNEL (The 'Filter'):")
for row in kernel:
    print(" ".join(f"{int(val):3d}" for val in row))

print("\n" + "="*60)
print("STEP 2: THE SLIDING WINDOW (CONVOLUTION IN ACTION)")
print("="*60)

# --- 3. HELPER FOR REPETITIVE PRINTING ---
def show_conv_step(row_idx, col_idx, img, k):
    # Extract the window
    window = img[row_idx:row_idx+3, col_idx:col_idx+3]
    # Multiply
    multiplied = window * k
    # Sum
    result = np.sum(multiplied)

    print(f"\n[STEP {col_idx+1}] SLIDING TO COLUMN {col_idx} (Output Position: {row_idx},{col_idx})")
    print("-" * 55)
    print(f"3x3 Image Window at [{row_idx}:{row_idx+3}, {col_idx}:{col_idx+3}]:")
    print(window)
    print("\nMultiplied by Kernel (Window * Kernel):")
    print(multiplied)
    print(f"--> FINAL SUM (MAC Result) = {result}")

# --- 4. THE THREE STEPS OF THE SLIDE ---
show_conv_step(0, 0, image, kernel)
show_conv_step(0, 1, image, kernel)
show_conv_step(0, 2, image, kernel)

# --- 5. THE FINAL RESULT ---
def simple_conv(img, k):
    h, w = img.shape
    kh, kw = k.shape
    res = np.zeros((h-kh+1, w-kw+1))
    for r in range(h-kh+1):
        for c in range(w-kw+1):
            res[r, c] = np.sum(img[r:r+kh, c:c+kw] * k)
    return res

feature_map = simple_conv(image, kernel)

print("\n" + "="*60)
print("STEP 3: THE FINAL FEATURE MAP (RESULT OF ALL SLIDES)")
print("="*60)
for row in feature_map:
    print(" ".join(f"{int(val):3d}" for val in row))

STEP 1: THE INPUT DATA & KERNEL

FULL 10x10 INPUT IMAGE:
  0   0   0   0  10  10   0   0   0   0
  0   0   0   0  10  10   0   0   0   0
  0   0   0   0  10  10   0   0   0   0
  0   0   0   0  10  10   0   0   0   0
  0   0   0   0  10  10   0   0   0   0
  0   0   0   0  10  10   0   0   0   0
  0   0   0   0  10  10   0   0   0   0
  0   0   0   0  10  10   0   0   0   0
  0   0   0   0  10  10   0   0   0   0
  0   0   0   0  10  10   0   0   0   0

3x3 CONVOLUTION KERNEL (The 'Filter'):
 -1   0   1
 -1   0   1
 -1   0   1

STEP 2: THE SLIDING WINDOW (CONVOLUTION IN ACTION)

[STEP 1] SLIDING TO COLUMN 0 (Output Position: 0,0)
-------------------------------------------------------
3x3 Image Window at [0:3, 0:3]:
[[0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]]

Multiplied by Kernel (Window * Kernel):
[[-0.  0.  0.]
 [-0.  0.  0.]
 [-0.  0.  0.]]
--> FINAL SUM (MAC Result) = 0.0

[STEP 2] SLIDING TO COLUMN 1 (Output Position: 0,1)
-------------------------------------------------------
3x3 Imag

In [2]:
# --- 6. APPLY ReLU (Rectified Linear Unit) ---
# Convention: Apply ReLU immediately after Convolution to remove negative "noise"
relu_feature_map = np.maximum(0, feature_map)

print("\n" + "="*60)
print("STEP 4: ReLU ACTIVATION (Removing Negative Values)")
print("="*60)
print("Notice how the -30s (falling edges) are now 0:")
for row in relu_feature_map:
    print(" ".join(f"{int(val):3d}" for val in row))

# --- 7. MAX POOLING (2x2 Window, Stride 2) ---
def apply_max_pooling(f_map):
    h, w = f_map.shape
    pooled_output = np.zeros((h // 2, w // 2))
    for r in range(0, h, 2):
        for c in range(0, w, 2):
            window = f_map[r:r+2, c:c+2]
            pooled_output[r // 2, c // 2] = np.max(window)
    return pooled_output

pooled_result = apply_max_pooling(relu_feature_map)

print("\n" + "="*60)
print("STEP 5: MAX POOLING (8x8 -> 4x4 Compression)")
print("="*60)
for row in pooled_result:
    print(" ".join(f"{int(val):3d}" for val in row))

# --- 8. FLATTENING ---
# Converting the 2D grid into a 1D vector for the final classifier
final_vector = pooled_result.flatten()

print("\n" + "="*60)
print("STEP 6: FLATTENING (The Final 1D Vector)")
print("="*60)
print(f"Vector Length: {len(final_vector)}")
print("Vector Content:")
print(final_vector)


STEP 4: ReLU ACTIVATION (Removing Negative Values)
Notice how the -30s (falling edges) are now 0:
  0   0  30  30   0   0   0   0
  0   0  30  30   0   0   0   0
  0   0  30  30   0   0   0   0
  0   0  30  30   0   0   0   0
  0   0  30  30   0   0   0   0
  0   0  30  30   0   0   0   0
  0   0  30  30   0   0   0   0
  0   0  30  30   0   0   0   0

STEP 5: MAX POOLING (8x8 -> 4x4 Compression)
  0  30   0   0
  0  30   0   0
  0  30   0   0
  0  30   0   0

STEP 6: FLATTENING (The Final 1D Vector)
Vector Length: 16
Vector Content:
[ 0. 30.  0.  0.  0. 30.  0.  0.  0. 30.  0.  0.  0. 30.  0.  0.]


In [3]:
import numpy as np

# --- 1. THE DATA (From your previous Pooling/Flattening step) ---
# This is the 1D vector: [0, 30, 0, 0, 0, 30, 0, 0, 0, 30, 0, 0, 0, 30, 0, 0]
# (Assuming ReLU was applied before pooling)
final_vector = np.array([0, 30, 0, 0, 0, 30, 0, 0, 0, 30, 0, 0, 0, 30, 0, 0])

# --- 2. THE LEARNED WEIGHTS (Templates) ---
# We define what the 'Ideal' Vertical and Horizontal bars look like in 1D.
# In a real CNN, the network "learns" these numbers through backpropagation.

# Vertical Template: High weights where the 2nd column of a 4x4 grid would be
# Indices: 1, 5, 9, 13
v_weights = np.array([
    0, 1, 0, 0,
    0, 1, 0, 0,
    0, 1, 0, 0,
    0, 1, 0, 0
])

# Horizontal Template: High weights where a middle row would be
# Indices: 4, 5, 6, 7
h_weights = np.array([
    0, 0, 0, 0,
    1, 1, 1, 1,
    0, 0, 0, 0,
    0, 0, 0, 0
])

# --- 3. THE INFERENCE (The 'Brain' Step) ---
# We calculate the Dot Product: Sum(Vector * Weights)
v_score = np.dot(final_vector, v_weights)
h_score = np.dot(final_vector, h_weights)

print("="*60)
print("FINAL STEP: CLASSIFICATION VIA DOT PRODUCT")
print("="*60)
print(f"Input Vector: {final_vector}")
print("-" * 60)
print(f"Vertical Template Match Score:   {v_score}")
print(f"Horizontal Template Match Score: {h_score}")
print("-" * 60)

# --- 4. THE DECISION ---
if v_score > h_score:
    print("RESULT >>> VERTICAL EDGE DETECTED")
elif h_score > v_score:
    print("RESULT >>> HORIZONTAL EDGE DETECTED")
else:
    print("RESULT >>> NO CLEAR PATTERN DETECTED")

FINAL STEP: CLASSIFICATION VIA DOT PRODUCT
Input Vector: [ 0 30  0  0  0 30  0  0  0 30  0  0  0 30  0  0]
------------------------------------------------------------
Vertical Template Match Score:   120
Horizontal Template Match Score: 30
------------------------------------------------------------
RESULT >>> VERTICAL EDGE DETECTED
